In [0]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("neo4jIngest").getOrCreate()

In [0]:
df=spark.sql("select * from ecom")

In [0]:
df.show()

+-------------------+-------------------+-------------------+-------------------+--------------------+-------+-------+-------------------+
|         event_time|           order_id|         product_id|        category_id|       category_code|  brand|  price|            user_id|
+-------------------+-------------------+-------------------+-------------------+--------------------+-------+-------+-------------------+
|2020-04-24 11:50:39|2294359932054536986|1515966223509089906|2268105426648170900|  electronics.tablet|samsung| 162.01|1515915625441993984|
|2020-04-24 11:50:39|2294359932054536986|1515966223509089906|2268105426648170900|  electronics.tablet|samsung| 162.01|1515915625441993984|
|2020-04-24 14:37:43|2294444024058086220|2273948319057183658|2268105430162997728|electronics.audio...| huawei|  77.52|1515915625447879434|
|2020-04-24 14:37:43|2294444024058086220|2273948319057183658|2268105430162997728|electronics.audio...| huawei|  77.52|1515915625447879434|
|2020-04-24 19:16:21|229458

In [0]:
cql="""
merge(o:Order{order_id:event.order_id,order_time:event.event_time})
merge(u:User{user_id:event.user_id})
merge(p:Product{brand:event.brand,product_id:event.product_id,price:event.price})
merge(b:Brand{name:event.brand})
merge(c:Category{category_id:event.category_id,category_code:event.category_code})
merge(u)-[:PLACED]->(o)
merge(o)-[:MADE_FOR]->(p)
merge(p)-[:BELONGS_TO]->(b)
merge(b)-[:HAS]->(c)
"""


In [0]:
from pyspark.sql.functions import col,when
for column in df.columns:
    df=df.withColumn(column,when(col(column).isNull()| (col(column)==""),"NA").otherwise(col(column)))

In [0]:
df.write.format("org.neo4j.spark.DataSource")\
    .option("url","neo4j+s://xxxxx")\
    .option("authentication.basic.username","neo4j")\
    .option("authentication.basic.password","xxxxx")\
    .option("batch.size","1000")\
    .option("query",cql)\
    .mode("overwrite")\
    .save()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:103)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$2(SequenceExecutionState.scala:103)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$2$adapted(SequenceExecutionState.scala:100)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:100)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:714)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:430)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:430)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio